In [1]:
import pandas as pd

df_diag = pd.read_csv("data/diagnosticos.csv")
df_mri = pd.read_csv("data/escaner.csv")
df_demo = pd.read_csv("data/demograficos.csv")
df_gene = pd.read_csv("data/genetica.csv")
df_mmse = pd.read_csv("data/mmse.csv")
df_adas = pd.read_csv("data/adas.csv", low_memory=False)
df_cdr = pd.read_csv("data/cdr.csv")

## FILTRADO DE COLUMNAS

In [2]:
df_demo = df_demo[["RID", "PTGENDER", "PTDOB", "PTEDUCAT"]]
df_gene = df_gene[["RID", "GENOTYPE"]]
df_diag = df_diag[["RID", "VISCODE2", "EXAMDATE", "DIAGNOSIS"]]
df_mmse = df_mmse[["RID", "VISCODE2", "VISDATE", "MMSCORE"]]
df_adas = df_adas[["RID", "VISCODE2", "VISDATE", "TOTSCORE", "TOTAL13"]]
df_cdr = df_cdr[["RID", "VISCODE2", "VISDATE", "CDGLOBAL", "CDRSB"]]

In [3]:
df_mri = df_mri[
    (df_mri["acquisition_type"] == "3D") &
    (df_mri["magnetic_field_strength"] == 3.0) &
    (df_mri["acquisition_plane"] == "Sagittal") &
    (df_mri["series_type"] == "T1w") &
    (~df_mri["series_description"].str.contains("FLAIR", case=False, na=False))
]

df_mri = df_mri[["image_id", "subject_id", "image_visit", "image_date", "series_description"]]

## DEMOGRAFIA

In [4]:
df_demo = df_demo.dropna()

df_demo = df_demo[df_demo["PTEDUCAT"] > 0]
df_demo["PTGENDER"] = df_demo["PTGENDER"].astype(int) # 1:hombre 2:mujer
df_demo["PTEDUCAT"] = df_demo["PTEDUCAT"].astype(int) # años de educacion
df_demo["PTDOB"] = pd.to_datetime(df_demo["PTDOB"], format="%m/%Y")

df_demo = df_demo.drop_duplicates(subset=["RID"], keep="first")
df_demo

,RID,PTGENDER,PTDOB,PTEDUCAT
0,2,1,1931-04-01,16
1,1,2,1944-12-01,18
2,3,1,1924-05-01,18
3,4,1,1938-01-01,10
4,5,1,1931-12-01,16
...,...,...,...,...
6178,12026,2,1970-05-01,10
6179,12029,2,1958-08-01,11
6180,12030,2,1961-03-01,14
6181,12033,1,1956-02-01,14


## GENES

In [5]:
df_gene = df_gene.dropna()

df_gene["APOE4CONT"] = df_gene["GENOTYPE"].apply(lambda x: x.count("4")) # numero de alelos 4
df_gene["APOE4CARRIER"] = (df_gene["APOE4CONT"] > 0).astype(int) # 1 si tiene alelo 4 0 si no

df_gene = df_gene.drop_duplicates(subset=["RID"], keep="last")
df_gene

,RID,GENOTYPE,APOE4CONT,APOE4CARRIER
0,2,3/3,0,0
1,3,3/4,1,1
2,4,3/3,0,0
3,5,3/3,0,0
4,7,3/4,1,1
...,...,...,...,...
3211,10326,4/4,2,1
3212,10771,4/4,2,1
3213,10787,3/4,1,1
3214,10360,3/3,0,0


## DIAGNOSTICO

In [6]:
df_diag = df_diag.dropna()

df_diag = df_diag[df_diag["DIAGNOSIS"].isin([1, 2, 3])]
df_diag["DIAGNOSIS"] = df_diag["DIAGNOSIS"].astype(int)
df_diag["EXAMDATE"] = pd.to_datetime(df_diag["EXAMDATE"], format="%Y-%m-%d")
df_diag["VISCODE2"] = df_diag["VISCODE2"].replace({"sc": "bl"})
df_diag = df_diag[df_diag["VISCODE2"].isin(["bl", "m12", "m24"])]

df_diag = df_diag.sort_values(by=["RID", "VISCODE2", "EXAMDATE"])
df_diag = df_diag.drop_duplicates(subset=["RID", "VISCODE2"], keep="last")
df_diag

,RID,VISCODE2,EXAMDATE,DIAGNOSIS
0,2,bl,2005-09-29,1
1,3,bl,2005-09-30,3
493,3,m12,2006-09-13,3
1893,3,m24,2007-09-12,3
16,4,bl,2005-11-08,2
...,...,...,...,...
15552,10891,bl,2025-09-10,2
15303,10893,bl,2025-07-03,2
15561,10894,bl,2025-09-12,2
15659,10895,bl,2025-10-10,2


## MMSE

In [7]:
df_mmse = df_mmse.dropna()

df_mmse = df_mmse[
    (df_mmse["MMSCORE"] >= 0) &
    (df_mmse["MMSCORE"] <= 30)
]
df_mmse["MMSCORE"] = df_mmse["MMSCORE"].astype(int)
df_mmse["VISDATE"] = pd.to_datetime(df_mmse["VISDATE"], format="%Y-%m-%d")
df_mmse["VISCODE2"] = df_mmse["VISCODE2"].replace({"sc": "bl"})
df_mmse = df_mmse[df_mmse["VISCODE2"].isin(["bl", "m12", "m24"])]


df_mmse = df_mmse.sort_values(by=["RID", "VISCODE2", "VISDATE"])
df_mmse = df_mmse.drop_duplicates(subset=["RID", "VISCODE2"], keep="last")
df_mmse

,RID,VISCODE2,VISDATE,MMSCORE
0,2,bl,2005-08-17,28
2,3,bl,2005-08-18,20
874,3,m12,2006-09-12,17
2333,3,m24,2007-09-12,19
3,4,bl,2005-08-18,27
...,...,...,...,...
14351,10891,bl,2025-07-03,28
14353,10893,bl,2025-07-03,27
14359,10894,bl,2025-07-08,27
14366,10895,bl,2025-07-09,28


## ADAS

In [8]:
df_adas = df_adas.dropna()

df_adas = df_adas[
    (df_adas["TOTSCORE"] >= 0) & (df_adas["TOTSCORE"] <= 70) &
    (df_adas["TOTAL13"] >= 0) & (df_adas["TOTAL13"] <= 85)
]
df_adas["VISDATE"] = pd.to_datetime(df_adas["VISDATE"], format="%Y-%m-%d")
df_adas = df_adas[df_adas["VISCODE2"].isin(["bl", "m12", "m24"])]

df_adas = df_adas.sort_values(by=["RID", "VISCODE2", "VISDATE"])
df_adas = df_adas.drop_duplicates(subset=["RID", "VISCODE2"], keep="last")
df_adas

,RID,VISCODE2,VISDATE,TOTSCORE,TOTAL13
3823,2,bl,2005-09-08,10.67,18.67
3820,3,bl,2005-09-12,22.00,31.00
3818,3,m12,2006-09-12,24.00,35.00
3817,3,m24,2007-09-12,25.67,37.67
3816,4,bl,2005-11-08,14.33,21.33
...,...,...,...,...,...
12785,10888,bl,2025-08-07,13.00,23.00
12730,10891,bl,2025-09-10,11.33,18.33
12737,10894,bl,2025-09-12,19.67,31.67
12773,10895,bl,2025-10-06,2.67,6.67


## CDR

In [9]:
df_cdr = df_cdr.dropna()

df_cdr = df_cdr[
    (df_cdr["CDGLOBAL"].isin([0.0, 0.5, 1.0, 2.0, 3.0])) &
    (df_cdr["CDRSB"] >= 0.0) & (df_cdr["CDRSB"] <= 18.0)
]
df_cdr["VISDATE"] = pd.to_datetime(df_cdr["VISDATE"], format="%Y-%m-%d")
df_cdr["VISCODE2"] = df_cdr["VISCODE2"].replace({"sc": "bl"})
df_cdr = df_cdr[df_cdr["VISCODE2"].isin(["bl", "m12", "m24"])]

df_cdr = df_cdr.sort_values(by=["RID", "VISCODE2", "VISDATE"])
df_cdr = df_cdr.drop_duplicates(subset=["RID", "VISCODE2"], keep="last")
df_cdr

,RID,VISCODE2,VISDATE,CDGLOBAL,CDRSB
0,2,bl,2005-08-17,0.0,0.0
1,3,bl,2005-08-18,1.0,4.5
815,3,m12,2006-09-12,0.5,3.5
2269,3,m24,2007-09-12,1.0,8.0
2,4,bl,2005-08-18,0.5,1.0
...,...,...,...,...,...
14940,12026,bl,2026-07-27,0.0,0.5
14948,12028,bl,2026-07-30,0.5,2.5
14942,12029,bl,2026-08-03,0.5,0.5
14943,12030,bl,2026-08-04,0.0,0.0


## MRI

In [10]:
df_mri = df_mri.dropna()

df_mri["RID"] = df_mri["subject_id"].apply(lambda x: int(str(x).split("_")[-1]))
df_mri["image_date"] = pd.to_datetime(df_mri["image_date"], format="%Y-%m-%d")
df_mri["image_visit"] = df_mri["image_visit"].replace({"scmri": "bl", "sc": "bl"})
df_mri = df_mri[df_mri["image_visit"].isin(["bl", "m12", "m24"])]

df_mri = df_mri.sort_values(by=["RID", "image_visit", "image_date", "image_id"])
df_mri = df_mri.drop_duplicates(subset=["RID", "image_visit"], keep="last")
df_mri = df_mri[["RID", "subject_id", "image_visit", "image_id", "image_date", "series_description"]]
df_mri

,RID,subject_id,image_visit,image_id,image_date,series_description
2874,30,023_S_0030,bl,8249,2005-10-26,MPRAGE Repeat
77906,30,023_S_0030,m12,26207,2006-10-12,MPRAGE
2885,30,023_S_0030,m24,82080,2007-11-13,MPRAGE
2902,31,023_S_0031,bl,8299,2005-10-28,MPRAGE Repeat
2914,31,023_S_0031,m12,28030,2006-10-30,MPRAGE
...,...,...,...,...,...,...
89205,10895,404_S_10895,bl,11418504,2025-08-14,Accelerated Sagittal MPRAGE (MSV22)
89330,10898,404_S_10898,bl,11427820,2025-08-27,Accelerated Sagittal MPRAGE (MSV22)
92508,12011,013_S_12011,bl,11711017,2026-06-23,Accelerated Sagittal MPRAGE (MSV21)
92343,12013,013_S_12013,bl,11687285,2026-06-30,Accelerated Sagittal MPRAGE (MSV21)


## MERGE GLOBAL

In [11]:
df_diag = df_diag.rename(columns={"EXAMDATE": "VISDATE_DIAG"})
df_mmse = df_mmse.rename(columns={"VISDATE": "VISDATE_MMSE"})
df_adas = df_adas.rename(columns={"VISDATE": "VISDATE_ADAS"})
df_cdr  = df_cdr.rename(columns={"VISDATE": "VISDATE_CDR"})
df_mri  = df_mri.rename(columns={"image_visit": "VISCODE2", "image_date": "VISDATE_MRI"})

df_tests = df_diag.merge(df_mmse, on=["RID", "VISCODE2"], how="inner")
df_tests = df_tests.merge(df_adas, on=["RID", "VISCODE2"], how="inner")
df_tests = df_tests.merge(df_cdr,  on=["RID", "VISCODE2"], how="inner")

df_in_t0_t1 = df_tests[df_tests["VISCODE2"].isin(["bl", "m12"])].merge(df_mri, on=["RID", "VISCODE2"], how="inner")
df_in_t2    = df_tests[df_tests["VISCODE2"] == "m24"]

df_long = pd.concat([df_in_t0_t1, df_in_t2], ignore_index=True)

rids_bl  = set(df_in_t0_t1[df_in_t0_t1["VISCODE2"] == "bl"]["RID"])
rids_m12 = set(df_in_t0_t1[df_in_t0_t1["VISCODE2"] == "m12"]["RID"])
rids_m24 = set(df_in_t2["RID"])

valid_rids = rids_bl.intersection(rids_m12).intersection(rids_m24)

df_static = df_demo.merge(df_gene, on="RID", how="inner")
valid_rids = valid_rids.intersection(set(df_static["RID"]))

df_long = df_long[df_long["RID"].isin(valid_rids)]
df1 = df_static.merge(df_long, on="RID", how="inner")


df1["order"] = df1["VISCODE2"].map({"bl": 0, "m12": 1, "m24": 2})
df1 = df1.sort_values(by=["RID", "order"]).drop(columns=["order"]).reset_index(drop=True)
df1


,RID,PTGENDER,PTDOB,PTEDUCAT,GENOTYPE,APOE4CONT,APOE4CARRIER,VISCODE2,VISDATE_DIAG,DIAGNOSIS,...,VISDATE_ADAS,TOTSCORE,TOTAL13,VISDATE_CDR,CDGLOBAL,CDRSB,subject_id,image_id,VISDATE_MRI,series_description
0,30,2,1925-10-01,19,3/3,0,0,bl,2005-10-20,2,...,2005-10-20,10.00,22.00,2005-09-29,0.5,1.0,023_S_0030,8249.0,2005-10-26,MPRAGE Repeat
1,30,2,1925-10-01,19,3/3,0,0,m12,2006-10-12,3,...,2006-10-12,17.00,26.00,2006-10-12,1.0,4.5,023_S_0030,26207.0,2006-10-12,MPRAGE
2,30,2,1925-10-01,19,3/3,0,0,m24,2007-11-13,3,...,2007-11-13,18.67,29.67,2007-11-13,1.0,7.0,NaN,NaN,NaT,NaN
3,31,2,1928-01-01,18,3/3,0,0,bl,2005-10-24,1,...,2005-10-24,1.67,1.67,2005-10-03,0.0,0.0,023_S_0031,8299.0,2005-10-28,MPRAGE Repeat
4,31,2,1928-01-01,18,3/3,0,0,m12,2006-10-30,1,...,2006-10-30,3.00,4.00,2006-10-30,0.0,0.0,023_S_0031,28030.0,2006-10-30,MPRAGE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2122,10151,1,1960-05-01,16,3/4,1,1,m12,2025-07-18,2,...,2025-07-14,5.33,11.33,2025-07-14,0.5,1.0,129_S_10151,11359887.0,2025-07-16,Accelerated Sag IR-FSPGR (MSV21)
2123,10151,1,1960-05-01,16,3/4,1,1,m24,2026-07-22,2,...,2026-07-01,4.33,7.33,2026-07-01,0.5,1.5,NaN,NaN,NaT,NaN
2124,10169,2,1968-02-01,15,3/4,1,1,bl,2024-10-31,3,...,2024-08-01,15.00,25.00,2024-07-16,0.5,3.5,941_S_10169,10883231.0,2024-07-17,Accelerated Sagittal MPRAGE (MSV21)
2125,10169,2,1968-02-01,15,3/4,1,1,m12,2025-08-12,3,...,2025-08-12,14.00,25.00,2025-08-12,1.0,5.5,941_S_10169,11452789.0,2025-09-26,Accelerated Sagittal MPRAGE (MSV21)


In [12]:
df1 = df1[df1["RID"] != 2007].reset_index(drop=True)

In [13]:
print(f"Número de pacientes que tienen los 3 instantes temporales: {df1["RID"].nunique()}")
print(f"Total de muestras (3*número de pacientes): {len(df1)}")
print(f"Distribución en el instante 'bl': \n{df1[df1["VISCODE2"] == "bl"]["DIAGNOSIS"].value_counts()}")
print(f"Distribución en el instante 'mes 12': \n{df1[df1["VISCODE2"] == "m12"]["DIAGNOSIS"].value_counts()}")
print(f"Distribución en el instante 'mes 24': \n{df1[df1["VISCODE2"] == "m24"]["DIAGNOSIS"].value_counts()}")

Número de pacientes que tienen los 3 instantes temporales: 708
Total de muestras (3*número de pacientes): 2124
Distribución en el instante 'bl': 
DIAGNOSIS
2    435
1    214
3     59
Name: count, dtype: int64
Distribución en el instante 'mes 12': 
DIAGNOSIS
2    396
1    220
3     92
Name: count, dtype: int64
Distribución en el instante 'mes 24': 
DIAGNOSIS
2    353
1    216
3    139
Name: count, dtype: int64


In [14]:
ids2 = df1["image_id"].dropna().astype(int).unique()
with open("data/ids1.txt", "w") as f:
    f.write(",".join([str(i) for i in ids2]))

In [16]:
df1.to_csv("data/preprocesados/merge1.csv", index=False)

## MERGE GLOBAL (MRI EN LAS 3)

In [17]:
df_diag = df_diag.rename(columns={"EXAMDATE": "VISDATE_DIAG"})
df_mmse = df_mmse.rename(columns={"VISDATE": "VISDATE_MMSE"})
df_adas = df_adas.rename(columns={"VISDATE": "VISDATE_ADAS"})
df_cdr = df_cdr.rename(columns={"VISDATE": "VISDATE_CDR"})
df_mri = df_mri.rename(columns={"image_visit": "VISCODE2", "image_date": "VISDATE_MRI"})

df_long = df_diag.merge(df_mmse, on=["RID", "VISCODE2"], how="inner")
df_long = df_long.merge(df_adas, on=["RID", "VISCODE2"], how="inner")
df_long = df_long.merge(df_cdr, on=["RID", "VISCODE2"], how="inner")
df_long = df_long.merge(df_mri, on=["RID", "VISCODE2"], how="inner")

visitas_por_paciente = df_long.groupby("RID")["VISCODE2"].nunique()
rids_completos = visitas_por_paciente[visitas_por_paciente == 3].index

df_long = df_long[df_long["RID"].isin(rids_completos)]

df_static = df_demo.merge(df_gene, on="RID", how="inner")
df2 = df_static.merge(df_long, on="RID", how="inner")

df2["order"] = df2["VISCODE2"].map({"bl": 0, "m12": 1, "m24": 2})
df2 = df2.sort_values(by=["RID", "order"]).drop(columns=["order"]).reset_index(drop=True)
df2

,RID,PTGENDER,PTDOB,PTEDUCAT,GENOTYPE,APOE4CONT,APOE4CARRIER,VISCODE2,VISDATE_DIAG,DIAGNOSIS,...,VISDATE_ADAS,TOTSCORE,TOTAL13,VISDATE_CDR,CDGLOBAL,CDRSB,subject_id,image_id,VISDATE_MRI,series_description
0,30,2,1925-10-01,19,3/3,0,0,bl,2005-10-20,2,...,2005-10-20,10.00,22.00,2005-09-29,0.5,1.0,023_S_0030,8249,2005-10-26,MPRAGE Repeat
1,30,2,1925-10-01,19,3/3,0,0,m12,2006-10-12,3,...,2006-10-12,17.00,26.00,2006-10-12,1.0,4.5,023_S_0030,26207,2006-10-12,MPRAGE
2,30,2,1925-10-01,19,3/3,0,0,m24,2007-11-13,3,...,2007-11-13,18.67,29.67,2007-11-13,1.0,7.0,023_S_0030,82080,2007-11-13,MPRAGE
3,31,2,1928-01-01,18,3/3,0,0,bl,2005-10-24,1,...,2005-10-24,1.67,1.67,2005-10-03,0.0,0.0,023_S_0031,8299,2005-10-28,MPRAGE Repeat
4,31,2,1928-01-01,18,3/3,0,0,m12,2006-10-30,1,...,2006-10-30,3.00,4.00,2006-10-30,0.0,0.0,023_S_0031,28030,2006-10-30,MPRAGE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1981,10144,2,1947-02-01,12,3/4,1,1,m12,2025-07-31,3,...,2025-07-31,16.67,29.67,2025-07-31,1.0,0.7,033_S_10144,11399151,2025-07-31,Accelerated Sagittal MPRAGE (MSV21)
1982,10144,2,1947-02-01,12,3/4,1,1,m24,2026-07-21,3,...,2026-07-21,17.00,29.00,2026-07-21,1.0,8.0,033_S_10144,11714669,2026-07-21,Accelerated Sagittal MPRAGE (MSV21)
1983,10151,1,1960-05-01,16,3/4,1,1,bl,2024-11-08,2,...,2024-07-18,5.00,10.00,2024-06-25,0.5,0.5,129_S_10151,10883423,2024-07-01,Accelerated Sag IR-FSPGR (MSV21)
1984,10151,1,1960-05-01,16,3/4,1,1,m12,2025-07-18,2,...,2025-07-14,5.33,11.33,2025-07-14,0.5,1.0,129_S_10151,11359887,2025-07-16,Accelerated Sag IR-FSPGR (MSV21)


In [18]:
df2 = df2[df2["RID"] != 2007].reset_index(drop=True)

In [19]:
print(f"Número de pacientes que tienen los 3 instantes temporales: {df2["RID"].nunique()}")
print(f"Total de muestras (3*número de pacientes): {len(df2)}")
print(f"Distribución en el instante 'bl': \n{df2[df2["VISCODE2"] == "bl"]["DIAGNOSIS"].value_counts()}")
print(f"Distribución en el instante 'mes 12': \n{df2[df2["VISCODE2"] == "m12"]["DIAGNOSIS"].value_counts()}")
print(f"Distribución en el instante 'mes 24': \n{df2[df2["VISCODE2"] == "m24"]["DIAGNOSIS"].value_counts()}")

Número de pacientes que tienen los 3 instantes temporales: 661
Total de muestras (3*número de pacientes): 1983
Distribución en el instante 'bl': 
DIAGNOSIS
2    409
1    201
3     51
Name: count, dtype: int64
Distribución en el instante 'mes 12': 
DIAGNOSIS
2    369
1    209
3     83
Name: count, dtype: int64
Distribución en el instante 'mes 24': 
DIAGNOSIS
2    332
1    204
3    125
Name: count, dtype: int64


In [20]:
ids2 = df2["image_id"].dropna().astype(int).unique()
with open("data/ids2.txt", "w") as f:
    f.write(",".join([str(i) for i in ids2]))